# Stage 03 — Instrument Alignment

Cross-correlate each Aeris instrument's CH4 against the Picarro (trusted reference) to find
per-file time lags, then apply them and write lag-shifted Parquet to `03_instrument_aligned/`.

**Pipeline position:** `02_standardized/` → **lag_offsets.json + aligned Parquet** → `03_instrument_aligned/`

## Cross-correlation sections

| Section | Instrument | Reference | Dates |
|---|---|---|---|
| A | Ultra 460 | Picarro | Feb 3–12 (WYO) |
| B | Ultra 321 | Picarro | WYO dates; MML dates auto-suggest 0s |
| C | Pico 017 WYO | Picarro | Feb 5–12 |
| D | Pico 017 MML + LGR | Ultra 321 MML | Jan 19–22, Feb 4, Mar 8–10 |

## Output
- `03_instrument_aligned/lag_offsets.json` — manifest with confirmed lags + rejected list
- `03_instrument_aligned/{instrument}/{subdir}/*.parquet` — lag-shifted aligned files

`no_coverage/` files from Stage 02 are excluded — they have uncorrectable timestamps.

## Workflow
1. Run **Imports**, **Config**, **Helpers**, and **Load Picarro** cells once
2. For each section: run **Auto-correlate**, then the **Widget** cell
3. In the widget:
   - **Commit Auto & Next** — save auto-detected lag, advance
   - **Set Lag & Commit** — adjust slider first, then save that value
   - **Mark Bad & Next** — exclude file from Stage 03 output
   - **Next ->** — skip without saving (file gets 0s lag with a warning at apply time)
4. Run **Save lag_offsets.json** after all sections are done
5. Run **Apply lags** to write the aligned Parquet files

In [ ]:
import json
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import ipywidgets as widgets
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display, HTML

sys.path.insert(0, str(Path().resolve().parent))
from paths import STAGE_02_DIR, STAGE_03_DIR, REPO_ROOT

# Force plotly figures to fill their container width
display(HTML('<style>.plotly-graph-div { width: 100% !important; }</style>'))

print('Imports OK')

In [ ]:
CH4_COL    = 'CH4_ppm'
MAX_LAG_S  = 600
RESAMPLE_S = 1

# MML-only deployment dates — no Picarro co-location, auto-correlate returns 0s
MML_DATE_TAGS = {'260119', '260120', '260121', '260122', '260202', '260204', '260308', '260310'}

PICARRO_DIR  = STAGE_02_DIR / 'WYO_picarro'
ULTRA460_DIR = STAGE_02_DIR / 'WYO_aerisultra460' / 'Raw'
ULTRA321_DIR = STAGE_02_DIR / 'LANL_aerisultra321' / 'Raw'
PICO017_DIR  = STAGE_02_DIR / 'LANL_aerispico017'  / 'Raw'
LGR_DIR      = STAGE_02_DIR / 'UOU_LGR'

print('Config OK')

In [ ]:
def load_parquet_ch4(path, col=CH4_COL):
    return pd.read_parquet(path, columns=[col])[col].dropna()

def resample_series(s, freq_s=RESAMPLE_S):
    s = s.resample(f'{freq_s}s').mean()
    return s.interpolate(method='time', limit=10)

def load_all_picarro(picarro_dir, col=CH4_COL):
    files = sorted(picarro_dir.glob('*.parquet'))
    combined = pd.concat([load_parquet_ch4(f, col) for f in files]).sort_index()
    combined = combined[~combined.index.duplicated(keep='first')]
    return resample_series(combined)

def load_ref_from_files(file_list, col=CH4_COL):
    combined = pd.concat([load_parquet_ch4(f, col) for f in file_list]).sort_index()
    combined = combined[~combined.index.duplicated(keep='first')]
    return resample_series(combined)

def cross_correlate(ref, sig, max_lag_s=MAX_LAG_S, freq_s=RESAMPLE_S):
    """
    Return lag in seconds. Positive = sig timestamps are behind UTC by that many seconds.
    Correction applied at output: df.index += pd.Timedelta(seconds=lag).
    """
    start = max(ref.index[0], sig.index[0])
    end   = min(ref.index[-1], sig.index[-1])
    if start >= end:
        print('    [warn] No overlapping time window — defaulting to 0s')
        return 0.0
    combined = pd.DataFrame({'r': ref[start:end], 's': sig[start:end]}).dropna()
    if len(combined) < 10:
        print(f'    [warn] Only {len(combined)} overlapping point(s) — defaulting to 0s')
        return 0.0
    r_arr    = (combined['r'] - combined['r'].mean()).values
    s_arr    = (combined['s'] - combined['s'].mean()).values
    max_samp = max_lag_s // freq_s
    corr     = np.correlate(r_arr, s_arr, mode='full')
    lags     = np.arange(-(len(r_arr) - 1), len(r_arr))
    mask     = np.abs(lags) <= max_samp
    return float(lags[mask][np.argmax(corr[mask])] * freq_s)

def auto_correlate(test_files, ref_data):
    """Cross-correlate each Parquet file against ref_data; print table; return suggestions dict."""
    suggestions = {}
    print(f"{'IDX':>4}  {'FILE':<55}  {'AUTO LAG':>10}")
    print('-' * 75)
    for i, f in enumerate(test_files):
        sig = resample_series(load_parquet_ch4(f))
        lag = cross_correlate(ref_data, sig)
        suggestions[f.stem] = lag
        tag = ' (MML)' if is_mml(f) else ''
        print(f'[{i:>2}]  {f.name:<55}  {lag:>+10.0f}s{tag}')
    return suggestions

# Ultra460 is WYO-only — never MML regardless of date.
# Ultra321 and Pico017 share MML_DATE_TAGS dates with WYO instruments (e.g. Feb 4),
# so instrument identity must be checked alongside the date tag.
WYO_ONLY_STEMS = {'Ultra100460'}

def is_mml(path):
    stem = path.stem
    instrument_prefix = stem.split('_')[0]
    if instrument_prefix in WYO_ONLY_STEMS:
        return False
    date_tag = stem.split('_')[1] if stem.count('_') >= 1 else ''
    return date_tag in MML_DATE_TAGS

def _git_info():
    try:
        h = subprocess.check_output(
            ['git', 'rev-parse', 'HEAD'], cwd=str(REPO_ROOT), text=True
        ).strip()
        dirty = subprocess.call(['git', 'diff', '--quiet'], cwd=str(REPO_ROOT)) != 0
        return h, dirty
    except Exception:
        return 'unknown', False

def save_lag_offsets():
    """Write current review state to lag_offsets.json. Called automatically after every commit."""
    STAGE_03_DIR.mkdir(parents=True, exist_ok=True)
    g = globals()
    pico_conf = {**g.get('pico_wyo_confirmed', {}), **g.get('pico_mml_confirmed', {})}
    pico_rej  = g.get('pico_wyo_rejected', set()) | g.get('pico_mml_rejected', set())

    def _lags(conf, rej):
        return {k: v for k, v in conf.items() if k not in rej}

    git_hash, git_dirty = _git_info()
    manifest = {
        'stage':     '03_instrument_alignment',
        'run_utc':   datetime.now(timezone.utc).isoformat(),
        'git_hash':  git_hash,
        'git_dirty': git_dirty,
        'lags': {
            'WYO_aerisultra460':  _lags(g.get('u460_confirmed', {}), g.get('u460_rejected', set())),
            'LANL_aerisultra321': _lags(g.get('u321_confirmed', {}), g.get('u321_rejected', set())),
            'LANL_aerispico017':  _lags(pico_conf, pico_rej),
            'UOU_LGR':            _lags(g.get('lgr_confirmed',  {}), g.get('lgr_rejected',  set())),
        },
        'rejected': {
            'WYO_aerisultra460':  sorted(g.get('u460_rejected', set())),
            'LANL_aerisultra321': sorted(g.get('u321_rejected', set())),
            'LANL_aerispico017':  sorted(pico_rej),
            'UOU_LGR':            sorted(g.get('lgr_rejected',  set())),
        },
    }
    with open(STAGE_03_DIR / 'lag_offsets.json', 'w') as fh:
        json.dump(manifest, fh, indent=2)

print('Data helpers loaded.')

In [ ]:
def make_review_widget(ref_data, test_files, test_name, suggestions, confirmed, rejected,
                       ref_name='Picarro (ref)', save_fn=None):
    """
    Interactive review widget. Modifies `confirmed` and `rejected` in-place.
    Calls save_fn() after every Commit or Mark Bad action (pass save_lag_offsets).

    Plotly figure for zooming/panning. Lag slider initialises to the auto-detected value.

    Buttons:
      Commit & Next   — save current slider value (adjust first if needed), advance
      Mark Bad & Next — send to bad/ subdir, advance
      Next ->         — skip without saving
    """
    if not test_files:
        print(f'{test_name}: no files to review')
        return

    state = {'idx': 0}

    # ── Plotly figure ──────────────────────────────────────────────────────────
    fig = go.FigureWidget(layout=go.Layout(
        autosize=True,
        height=400,
        margin=dict(l=55, r=15, t=62, b=40),
        yaxis=dict(title=CH4_COL),
        legend=dict(x=1, y=1, xanchor='right', font=dict(size=10)),
        hovermode='x unified',
    ))
    # Reference: medium gray, moderate width — context, not the focus
    fig.add_scatter(name=ref_name,  line=dict(color='#666666', width=1.5), opacity=0.75)
    # Test instrument: vivid orange, thicker — this is what we're aligning
    fig.add_scatter(name=test_name, line=dict(color='#E67E22', width=2.5))

    # ── Lag slider ─────────────────────────────────────────────────────────────
    lag_slider = widgets.FloatSlider(
        value=0.0, min=-MAX_LAG_S, max=MAX_LAG_S, step=0.1,
        description='Lag (s):',
        continuous_update=True,
        readout_format='.1f',
        layout=widgets.Layout(width='100%'),
        style={'description_width': '60px'},
    )

    # ── Buttons ────────────────────────────────────────────────────────────────
    btn_prev   = widgets.Button(description='← Prev',           layout=widgets.Layout(width='85px'))
    btn_next   = widgets.Button(description='Next →',           layout=widgets.Layout(width='85px'))
    btn_commit = widgets.Button(description='Commit & Next',    button_style='success', layout=widgets.Layout(width='140px'))
    btn_bad    = widgets.Button(description='Mark Bad & Next',  button_style='danger',  layout=widgets.Layout(width='150px'))

    # ── Log ────────────────────────────────────────────────────────────────────
    log = widgets.Output(layout=widgets.Layout(
        height='90px', overflow_y='auto',
        border='1px solid #ddd', padding='4px',
    ))

    # ── Plot update ────────────────────────────────────────────────────────────
    def _update_fig(idx, lag_s):
        if idx >= len(test_files):
            with fig.batch_update():
                fig.data[0].x = []
                fig.data[0].y = []
                fig.data[1].x = []
                fig.data[1].y = []
            fig.layout.title = dict(
                text=(f'{test_name} — complete  '
                      f'({len(confirmed)} confirmed, {len(rejected)} rejected)'),
                y=0.97, yanchor='top',
            )
            return

        f        = test_files[idx]
        key      = f.stem
        auto_lag = suggestions.get(key, 0.0)

        try:
            sig_raw = resample_series(load_parquet_ch4(f))
        except Exception as e:
            fig.layout.title = dict(text=f'ERROR loading {f.name}: {e}')
            return

        shifted_idx = sig_raw.index + pd.Timedelta(seconds=lag_s)
        t0 = sig_raw.index[0]  - pd.Timedelta(hours=1)
        t1 = sig_raw.index[-1] + pd.Timedelta(hours=1)
        ref_win = ref_data[t0:t1]

        with fig.batch_update():
            fig.data[0].x = ref_win.index
            fig.data[0].y = ref_win.values
            fig.data[1].x = shifted_idx
            fig.data[1].y = sig_raw.values
            fig.data[1].name = f'{test_name} (lag={lag_s:+.1f}s)'

        status = ''
        if key in confirmed:
            status = f'  ✓ {confirmed[key]:+.1f}s'
        elif key in rejected:
            status = '  ✗ bad'

        mml_tag = ' · MML' if is_mml(f) else ''

        t_start  = sig_raw.index[0].strftime('%H:%M')
        t_end    = sig_raw.index[-1].strftime('%H:%M')
        n_rows   = len(sig_raw)
        n_ref    = int(ref_win.notna().sum())
        subtitle = (f'{n_rows:,} rows · {t_start}–{t_end} UTC'
                    f' · ref: {n_ref:,} rows in window')

        fig.layout.title = dict(
            text=(f'[{idx+1}/{len(test_files)}]  {f.name}'
                  f'  auto={auto_lag:+.1f}s{mml_tag}{status}'
                  f'<br><sup>{subtitle}</sup>'),
            y=0.97, yanchor='top',
        )

    def go_to(idx):
        if 0 <= idx < len(test_files):
            lag_slider.value = suggestions.get(test_files[idx].stem, 0.0)
        _update_fig(idx, lag_slider.value)

    lag_slider.observe(lambda change: _update_fig(state['idx'], change['new']), names='value')

    # ── Button callbacks ───────────────────────────────────────────────────────
    def on_prev(_):
        state['idx'] = max(0, state['idx'] - 1)
        go_to(state['idx'])

    def on_next(_):
        state['idx'] += 1
        go_to(state['idx'])

    def on_commit(_):
        if state['idx'] >= len(test_files): return
        key = test_files[state['idx']].stem
        lag = round(lag_slider.value, 1)
        confirmed[key] = lag
        rejected.discard(key)
        if save_fn: save_fn()
        with log: print(f'COMMITTED  {key}  {lag:+.1f}s')
        state['idx'] += 1
        go_to(state['idx'])

    def on_bad(_):
        if state['idx'] >= len(test_files): return
        key = test_files[state['idx']].stem
        rejected.add(key)
        confirmed.pop(key, None)
        if save_fn: save_fn()
        with log: print(f'REJECTED   {key}')
        state['idx'] += 1
        go_to(state['idx'])

    btn_prev.on_click(on_prev)
    btn_next.on_click(on_next)
    btn_commit.on_click(on_commit)
    btn_bad.on_click(on_bad)

    btn_row = widgets.HBox(
        [btn_prev, btn_bad, btn_commit, btn_next],
        layout=widgets.Layout(gap='6px', margin='4px 0'),
    )
    display(widgets.VBox(
        [fig, lag_slider, btn_row, log],
        layout=widgets.Layout(width='100%'),
    ))
    go_to(0)

print('Widget helper loaded.')

In [ ]:
print('Loading Picarro reference (187 files — may take ~30s)...')
picarro_ref = load_all_picarro(PICARRO_DIR)
print(f'\nPicarro (CH4_ppm): {len(picarro_ref):,} samples')
print(f'  Range: {picarro_ref.index[0]}  ->  {picarro_ref.index[-1]}')

---
## A — Ultra 460 vs Picarro

Ultra 460 is WYO-only (Feb 3–12). All 44 files should have good Picarro overlap.

Run auto-correlate, then the widget. Click **Commit Auto & Next** for each file, or adjust
with the **Lag** slider and use **Set Lag & Commit** to override.

In [ ]:
u460_files = sorted(ULTRA460_DIR.glob('*.parquet'))
print(f'Ultra 460: {len(u460_files)} files\n')
u460_suggestions = auto_correlate(u460_files, picarro_ref)

In [ ]:
if 'u460_confirmed' not in dir():
    u460_confirmed = {}
if 'u460_rejected' not in dir():
    u460_rejected = set()

make_review_widget(picarro_ref, u460_files, 'Ultra460',
                   u460_suggestions, u460_confirmed, u460_rejected,
                   save_fn=save_lag_offsets)

---
## B — Ultra 321 vs Picarro

Ultra 321 WYO dates (Feb 3, Feb 5–12): have Picarro overlap, auto-correlate should find a lag.  
Ultra 321 MML dates (Jan 19–22, Feb 2, Feb 4, Mar 8–10): no Picarro — auto-suggest 0s.

For MML files the Stage 01 Toughbook correction already removed the ~6-hour offset. Options:
- Commit at 0s (conservative default)
- Use **Set Lag & Commit** with the same lag observed in WYO files (same instrument, consistent hardware)
- Mark bad if the file looks unusable

In [ ]:
u321_files = sorted(ULTRA321_DIR.glob('*.parquet'))
print(f'Ultra 321: {len(u321_files)} files')
print('MML dates (no Picarro): Jan 19-22, Feb 2, Feb 4, Mar 8-10 — auto-suggest 0s\n')
u321_suggestions = auto_correlate(u321_files, picarro_ref)

In [ ]:
if 'u321_confirmed' not in dir():
    u321_confirmed = {}
if 'u321_rejected' not in dir():
    u321_rejected = set()

make_review_widget(picarro_ref, u321_files, 'Ultra321',
                   u321_suggestions, u321_confirmed, u321_rejected,
                   save_fn=save_lag_offsets)

---
## C — Pico 017 WYO dates vs Picarro

Pico 017 was on WYO only from Feb 5–12. MML dates (Jan 19–22, Feb 4, Mar 8–10) are handled
in Section D where Ultra 321 is used as the reference.

In [ ]:
pico_all = sorted(PICO017_DIR.glob('*.parquet'))
pico_wyo = [f for f in pico_all if not is_mml(f)]
pico_mml = [f for f in pico_all if     is_mml(f)]
print(f'Pico017 WYO dates: {len(pico_wyo)} files  (this section — vs Picarro)')
print(f'Pico017 MML dates: {len(pico_mml)} files  (Section D — vs Ultra321)\n')
pico_wyo_suggestions = auto_correlate(pico_wyo, picarro_ref)

In [ ]:
if 'pico_wyo_confirmed' not in dir():
    pico_wyo_confirmed = {}
if 'pico_wyo_rejected' not in dir():
    pico_wyo_rejected = set()

make_review_widget(picarro_ref, pico_wyo, 'Pico017',
                   pico_wyo_suggestions, pico_wyo_confirmed, pico_wyo_rejected,
                   save_fn=save_lag_offsets)

---
## D — Pico 017 MML dates + LGR vs Ultra 321

For MML deployments both instruments were corrected to the same Toughbook logger, so any lag
measured here is a genuine instrument-specific offset (inlet path, sampling rate, etc.).

LGR was MML-only on Mar 10. It has trusted timestamps but cross-correlating vs Ultra 321
measures any residual physical lag.

In [ ]:
u321_mml_files = [f for f in sorted(ULTRA321_DIR.glob('*.parquet')) if is_mml(f)]
print(f'Loading Ultra321 MML reference ({len(u321_mml_files)} files: Jan 19-22, Feb 2, Feb 4, Mar 8-10)...')
u321_mml_ref = load_ref_from_files(u321_mml_files)
print(f'\nUltra321 MML (CH4_ppm): {len(u321_mml_ref):,} samples')
print(f'  Range: {u321_mml_ref.index[0]}  ->  {u321_mml_ref.index[-1]}')
print(f'\nPico017 MML dates: {len(pico_mml)} files\n')
pico_mml_suggestions = auto_correlate(pico_mml, u321_mml_ref)

In [ ]:
if 'pico_mml_confirmed' not in dir():
    pico_mml_confirmed = {}
if 'pico_mml_rejected' not in dir():
    pico_mml_rejected = set()

make_review_widget(u321_mml_ref, pico_mml, 'Pico017-MML',
                   pico_mml_suggestions, pico_mml_confirmed, pico_mml_rejected,
                   ref_name='Ultra321 MML (ref)',
                   save_fn=save_lag_offsets)

In [ ]:
lgr_files = sorted(LGR_DIR.glob('*.parquet'))
print(f'LGR: {len(lgr_files)} file(s)  (Mar 10 MML — vs Ultra321 MML)\n')
lgr_suggestions = auto_correlate(lgr_files, u321_mml_ref)

In [ ]:
if 'lgr_confirmed' not in dir():
    lgr_confirmed = {}
if 'lgr_rejected' not in dir():
    lgr_rejected = set()

make_review_widget(u321_mml_ref, lgr_files, 'LGR',
                   lgr_suggestions, lgr_confirmed, lgr_rejected,
                   ref_name='Ultra321 MML (ref)',
                   save_fn=save_lag_offsets)

In [ ]:
pico_confirmed = {**pico_wyo_confirmed, **pico_mml_confirmed}
pico_rejected  = pico_wyo_rejected | pico_mml_rejected
print(f'Pico017 total: {len(pico_confirmed)} confirmed, {len(pico_rejected)} rejected')

---
## Save lag_offsets.json

Run once all widget review sections are done. Writes a single JSON to `03_instrument_aligned/`
that is the source of truth consumed by the Apply section below.

In [ ]:
pico_confirmed = {**pico_wyo_confirmed, **pico_mml_confirmed}
pico_rejected  = pico_wyo_rejected | pico_mml_rejected
print(f'Pico017 total: {len(pico_confirmed)} confirmed, {len(pico_rejected)} rejected')

save_lag_offsets()
lag_path = STAGE_03_DIR / 'lag_offsets.json'
print(f'\nFinal save -> {lag_path}')

with open(lag_path) as fh:
    saved = json.load(fh)

for inst, lags in saved['lags'].items():
    rej = saved['rejected'].get(inst, [])
    if lags or rej:
        print(f'\n  {inst}: {len(lags)} confirmed, {len(rej)} rejected')
        for stem, lag in sorted(lags.items()):
            print(f'    {stem:<55}  {lag:>+6.0f}s')

---
## Apply lags and write aligned Parquet

Reads `lag_offsets.json` and applies each confirmed lag to Raw, Eng, and Spectra/Spectralite files.
Writes lag-shifted Parquet to `03_instrument_aligned/{instrument}/{subdir}/`.

Every file ends up somewhere — nothing is silently dropped:

| File | Destination |
|---|---|
| Confirmed lag | `{subdir}/*.parquet` — lag-shifted |
| Skipped in widget (no entry) | `{subdir}/*.parquet` — lag = 0s, warning logged |
| Marked Bad in widget | `{subdir}/bad/*.parquet` — copied as-is, no lag applied |
| no_coverage (Stage 01 uncorrectable) | `{subdir}/bad_timestamp/*.parquet` — pass-through cell below |

`bad/` files are typically startup files or sessions where the signal was too noisy to align.
They're kept for reference but should not be used for cross-instrument analysis.

Set `APPLY_SPECTRA = False` to skip spectra files if the run is too slow (1034 cols each).

In [ ]:
import shutil

def raw_stem(path):
    """Strip Eng/spectra/spectralite suffix to get the Raw file stem for lag lookup."""
    s = path.stem
    for suffix in ('Eng', 'spectra', 'spectralite'):
        if s.endswith(suffix):
            return s[:-len(suffix)]
    return s

def apply_lag_to_parquet(src, lag_s, dst):
    df = pd.read_parquet(src)
    if lag_s != 0.0:
        df.index = df.index + pd.Timedelta(seconds=lag_s)
    dst.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(dst)
    return len(df)

def apply_instrument(instrument, subdirs, lags, rejected_stems, apply_spectra=True):
    src_inst = STAGE_02_DIR / instrument
    dst_inst = STAGE_03_DIR / instrument
    n_ok = n_bad = n_warn = 0

    for subdir in subdirs:
        if not apply_spectra and subdir in ('Spectra', 'Spectralite'):
            print(f'  [SKIP spectra]  {instrument}/{subdir}')
            continue
        src_dir = src_inst / subdir if subdir else src_inst
        if not src_dir.exists():
            continue
        for path in sorted(src_dir.glob('*.parquet')):
            rs = raw_stem(path)
            dst_base = dst_inst / subdir if subdir else dst_inst
            if rs in rejected_stems:
                dst_path = dst_base / 'bad' / path.name
                dst_path.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(path, dst_path)
                print(f'  [BAD]  {path.name:<55}  -> bad/')
                n_bad += 1
                continue
            lag_s = lags.get(rs)
            if lag_s is None:
                print(f'  [WARN no lag]  {instrument}/{subdir or ""}/{path.name} — using 0s')
                lag_s  = 0.0
                n_warn += 1
            dst_path = dst_base / path.name
            rows = apply_lag_to_parquet(path, lag_s, dst_path)
            print(f'  [OK]  {path.name:<55}  {lag_s:>+6.0f}s  [{rows:,} rows]')
            n_ok += 1

    print(f'  -> {instrument}: aligned={n_ok}, bad={n_bad}, warn={n_warn}')
    return {'ok': n_ok, 'bad': n_bad, 'warn': n_warn}

print('Apply helpers loaded.')

In [ ]:
APPLY_SPECTRA = True

INSTRUMENT_SUBDIRS = {
    'WYO_aerisultra460':  ['Raw', 'Eng', 'Spectralite'],
    'LANL_aerisultra321': ['Raw', 'Eng', 'Spectra'],
    'LANL_aerispico017':  ['Raw', 'Eng', 'Spectra'],
    'UOU_LGR':            [''],
}

with open(STAGE_03_DIR / 'lag_offsets.json') as fh:
    saved = json.load(fh)

apply_stats = {}
for inst, subdirs in INSTRUMENT_SUBDIRS.items():
    print(f'\n{"="*60}')
    print(f'  {inst}')
    print(f'{"="*60}')
    lags     = saved['lags'].get(inst, {})
    rejected = set(saved['rejected'].get(inst, []))
    stats    = apply_instrument(inst, subdirs, lags, rejected, apply_spectra=APPLY_SPECTRA)
    apply_stats[inst] = stats

apply_manifest = {
    'stage':         '03_apply',
    'run_utc':       datetime.now(timezone.utc).isoformat(),
    'git_hash':      saved['git_hash'],
    'git_dirty':     saved['git_dirty'],
    'apply_spectra': APPLY_SPECTRA,
    'instruments':   apply_stats,
}
apply_path = STAGE_03_DIR / 'apply_manifest.json'
with open(apply_path, 'w') as fh:
    json.dump(apply_manifest, fh, indent=2)

print(f'\nApply manifest -> {apply_path}')
print('Run the pass-through cell below to complete Stage 03.')

---
## Pass-through: no_coverage → bad_timestamp

Files that had no logger coverage in Stage 01 couldn't have their timestamps corrected — their
clocks were reading Mountain Time, not UTC. There is nothing Stage 03 can do to fix that, but
the data itself is still valid and worth keeping.

These files are copied unchanged from `02_standardized/{instrument}/{subdir}/no_coverage/` to
`03_instrument_aligned/{instrument}/{subdir}/bad_timestamp/`. The subdirectory rename makes the
consequence explicit: anyone looking at the Stage 03 output immediately knows these files have
unreliable timestamps, without needing to know what "no_coverage" means in Stage 01 terms.

The `ts_status` column inside each file still reads `'no_coverage'` (the cause). The directory
name `bad_timestamp/` conveys the effect. Stage 04 will write these rows to
`04_daily/{instrument}/bad_timestamp/YYYYMMDD.parquet` — separate from the aligned daily files.

In [ ]:
# Instruments that have no_coverage subdirs (LANL only — WYO/trusted have none)
NO_COVERAGE_SUBDIRS = {
    'LANL_aerisultra321': ['Raw', 'Eng', 'Spectra'],
    'LANL_aerispico017':  ['Raw', 'Eng', 'Spectra'],
}

passthrough_stats = {}
for inst, subdirs in NO_COVERAGE_SUBDIRS.items():
    n_ok = 0
    for subdir in subdirs:
        if not APPLY_SPECTRA and subdir == 'Spectra':
            print(f'  [SKIP spectra]  {inst}/{subdir}/no_coverage')
            continue
        src_dir = STAGE_02_DIR / inst / subdir / 'no_coverage'
        dst_dir = STAGE_03_DIR / inst / subdir / 'bad_timestamp'
        if not src_dir.exists():
            continue
        files = sorted(src_dir.glob('*.parquet'))
        if not files:
            continue
        dst_dir.mkdir(parents=True, exist_ok=True)
        for path in files:
            shutil.copy2(path, dst_dir / path.name)
            print(f'  [PASS]  {inst}/{subdir}/bad_timestamp/{path.name}')
            n_ok += 1
    passthrough_stats[inst] = {'copied': n_ok}
    print(f'  -> {inst}: {n_ok} files copied to bad_timestamp/')

# Append passthrough stats to the apply manifest
apply_manifest_path = STAGE_03_DIR / 'apply_manifest.json'
with open(apply_manifest_path) as fh:
    apply_manifest = json.load(fh)
apply_manifest['passthrough'] = passthrough_stats
with open(apply_manifest_path, 'w') as fh:
    json.dump(apply_manifest, fh, indent=2)

print(f'\nPass-through complete')
print(f'Manifest updated  -> {apply_manifest_path}')
print(f'Stage 03 complete -> {STAGE_03_DIR}')

---
## Pass-through: trusted instruments → Stage 03

Picarro, Sprinter, LANL_GPS, and LANL_Anem all carry UTC-accurate timestamps
(trusted epoch or GPS-synced) and need no lag correction. Copy them from Stage 02
to Stage 03 unchanged (lag = 0) so that Stage 03 is a complete, self-contained
aligned dataset for the analysis repo.


In [ ]:
TRUSTED_INSTRUMENTS = [
    'WYO_picarro',
    'WYO_sprinter',
    'LANL_GPS',
    'LANL_Anem',
]

trusted_stats = {}
for inst in TRUSTED_INSTRUMENTS:
    src_dir = STAGE_02_DIR / inst
    dst_dir = STAGE_03_DIR / inst
    files = sorted(src_dir.glob('*.parquet'))
    if not files:
        print(f'[WARN]  {inst} — no parquet files found in {src_dir}')
        trusted_stats[inst] = {'ok': 0, 'warn': 'no files'}
        continue
    n = len(files)
    n_ok = 0
    print(f'\n{"="*60}')
    print(f'  {inst}  ({n} files)')
    print(f'{"="*60}')
    for f in files:
        dst = dst_dir / f.name
        rows = apply_lag_to_parquet(f, 0.0, dst)
        print(f'  OK  {f.name}  [{rows:,} rows]')
        n_ok += 1
    trusted_stats[inst] = {'ok': n_ok}

print('\nTrusted pass-through complete.')
